In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import numpy as np
import json
import qiskit.qasm2
from qiskit.quantum_info import Statevector

# Import your viewer classes
from qc_interactive_education_package import InteractiveViewer, ChallengeViewer

# ==========================================
# 1. JSON INGESTION & PARSING PIPELINE
# ==========================================

def load_libraries(filepath="library.json"):
    """Reads the static JSON configuration and parses it into mathematical objects."""
    try:
        with open(filepath, 'r') as f:
            data = json.load(f)
    except FileNotFoundError:
        print(f"❌ Critical Error: Configuration file '{filepath}' not found.")
        return {}, {}
    except json.JSONDecodeError as e:
        print(f"❌ Critical Error: Invalid JSON syntax in '{filepath}'. Details: {e}")
        return {}, {}

    # Parse Algorithms (QASM -> QuantumCircuit)
    algos = {}
    for name, qasm_str in data.get("algorithms", {}).items():
        try:
            algos[name] = qiskit.qasm2.loads(qasm_str)
        except Exception as e:
            print(f"Warning: Failed to compile QASM for '{name}'. {e}")

    # Parse Challenges (Lists -> Complex Numpy Arrays)
    challenges = {}
    for name, chal_data in data.get("challenges", {}).items():
        try:
            # Reconstruct complex tensors from [real, imag] pairs or standard floats
            init_parsed = _parse_complex_array(chal_data["initial_state"])
            targ_parsed = _parse_complex_array(chal_data["target_state"])

            challenges[name] = {
                "num_qubits": chal_data["num_qubits"],
                "initial_state": init_parsed,
                "target_state": targ_parsed
            }
        except Exception as e:
             print(f"Warning: Failed to parse state array for '{name}'. {e}")

    return algos, challenges

def _parse_complex_array(state_list):
    """Safely converts generic JSON arrays into typed Python complex arrays."""
    if not state_list:
        return None
    # If standard 1D array of floats (purely real state)
    if isinstance(state_list[0], (int, float)):
        return [complex(x, 0.0) for x in state_list]
    # If 2D array of [real, imaginary] pairs
    elif isinstance(state_list[0], list):
        return [complex(x[0], x[1]) for x in state_list]
    raise ValueError("Unrecognized array format in JSON.")

# ==========================================
# 2. SPA ENTRY POINT APPLICATION
# ==========================================

class QuantumAppLauncher:
    """
    The Single Page Application (SPA) entry point.
    Structurally isolates Sandbox, Algorithms, and Challenges using a Tab interface.
    """
    def __init__(self):
        self.output = widgets.Output()

        # Load the external library data
        self.algos, self.challenges = load_libraries("library.json")

        # --- UI Assembly: Header ---
        self.title = widgets.HTML("<h1 style='text-align: center; color: #2c3e50;'>Quantum Education Suite</h1>")
        self.subtitle = widgets.HTML("<h4 style='text-align: center; color: #7f8c8d; margin-bottom: 20px;'>Select your learning environment</h4>")
        self.header = widgets.VBox([self.title, self.subtitle], layout={'width': '100%'})

        # --- Build Tabs ---
        self.tab = widgets.Tab(layout={'width': '500px', 'min_height': '250px'})

        self.tab_sandbox = self._build_sandbox_tab()
        self.tab_algo = self._build_algorithm_tab()
        self.tab_challenge = self._build_challenge_tab()

        self.tab.children = [self.tab_sandbox, self.tab_algo, self.tab_challenge]
        self.tab.titles = ('Sandbox', 'Algorithms', 'Challenges')

        self.menu_container = widgets.VBox(
            [self.header, self.tab],
            layout=widgets.Layout(align_items='center', justify_content='center', width='100%', margin='30px 0px')
        )

    # --- UI Generators ---

    def _build_sandbox_tab(self):
        self.sb_qubits = widgets.Dropdown(options=[1, 2, 3, 4, 5, 6], value=3, description='Qubits:')
        self.sb_circuit = widgets.Checkbox(value=True, description='Show Circuit UI', indent=False)

        self.sb_states = {
            "|0...0⟩ (Ground State)": None,
            "|+...+⟩ (Equal Superposition)": "superposition"
        }
        self.sb_initial = widgets.Dropdown(options=list(self.sb_states.keys()), value="|0...0⟩ (Ground State)", description='Initial State:')

        btn = widgets.Button(description="Launch Sandbox", layout={'width': '100%', 'margin': '15px 0px 0px 0px'})
        btn.style.button_color = '#3498db'; btn.style.text_color = 'white'; btn.style.font_weight = 'bold'
        btn.on_click(self._launch_sandbox)

        return widgets.VBox([self.sb_qubits, self.sb_initial, self.sb_circuit, btn], layout={'padding': '20px', 'align_items': 'center'})

    def _build_algorithm_tab(self):
        # Handle empty library edge case gracefully
        options = list(self.algos.keys()) if self.algos else ["No algorithms loaded"]
        self.algo_dropdown = widgets.Dropdown(options=options, description='Algorithm:', layout={'width': '350px'})
        self.algo_circuit = widgets.Checkbox(value=True, description='Show Circuit UI', indent=False)

        btn = widgets.Button(description="Study Algorithm", layout={'width': '100%', 'margin': '15px 0px 0px 0px'})
        btn.style.button_color = '#9b59b6'; btn.style.text_color = 'white'; btn.style.font_weight = 'bold'
        btn.disabled = not bool(self.algos)
        btn.on_click(self._launch_algorithm)

        return widgets.VBox([self.algo_dropdown, self.algo_circuit, btn], layout={'padding': '20px', 'align_items': 'center'})

    def _build_challenge_tab(self):
        options = list(self.challenges.keys()) if self.challenges else ["No challenges loaded"]
        self.chal_dropdown = widgets.Dropdown(options=options, description='Challenge:', layout={'width': '400px'})
        self.chal_circuit = widgets.Checkbox(value=True, description='Show Circuit UI', indent=False)

        btn = widgets.Button(description="Start Challenge", layout={'width': '100%', 'margin': '15px 0px 0px 0px'})
        btn.style.button_color = '#e67e22'; btn.style.text_color = 'white'; btn.style.font_weight = 'bold'
        btn.disabled = not bool(self.challenges)
        btn.on_click(self._launch_challenge)

        return widgets.VBox([self.chal_dropdown, self.chal_circuit, btn], layout={'padding': '20px', 'align_items': 'center'})

    # --- Routing Logic ---

    def _generate_sb_array(self, num_qubits, state_type):
        if state_type is None: return None
        dim = 2 ** num_qubits
        if state_type == "superposition": return [1.0 / np.sqrt(dim)] * dim

    def _launch_sandbox(self, b):
        num_qubits = self.sb_qubits.value
        state_key = self.sb_initial.value
        initial_state = self._generate_sb_array(num_qubits, self.sb_states[state_key])

        with self.output:
            clear_output(wait=True)
            viewer = InteractiveViewer(
                num_qubits=num_qubits,
                initial_state=initial_state,
                show_circuit=self.sb_circuit.value
            )
            viewer.display()

    def _launch_algorithm(self, b):
        algo_name = self.algo_dropdown.value
        qc_algo = self.algos[algo_name]

        # Mathematically deduce the target state by simulating the exact algorithm
        initial_state = ([1.0] + [0.0] * ((2**qc_algo.num_qubits) - 1))
        target_sv = Statevector.from_instruction(qc_algo).data.tolist()

        with self.output:
            clear_output(wait=True)
            viewer = ChallengeViewer(
                num_qubits=qc_algo.num_qubits,
                initial_state=initial_state,
                target_state=target_sv,
                preloaded_circuit=qc_algo,
                show_circuit=self.algo_circuit.value
            )
            viewer.display()

    def _launch_challenge(self, b):
        chal_name = self.chal_dropdown.value
        chal_data = self.challenges[chal_name]

        with self.output:
            clear_output(wait=True)
            viewer = ChallengeViewer(
                num_qubits=chal_data["num_qubits"],
                initial_state=chal_data["initial_state"],
                target_state=chal_data["target_state"],
                preloaded_circuit=None,
                show_circuit=self.chal_circuit.value
            )
            viewer.display()

    def display(self):
        with self.output:
            clear_output(wait=True)
            display(self.menu_container)
        display(self.output)

# Instantiate and render
app = QuantumAppLauncher()
app.display()